In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

In [2]:
df = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/PINAGSAMASAMA.xlsx', sheet_name='PINAGSAMASAMA')

In [3]:
df.head(5)

,datetime,main.aqi,components.co,components.no,components.no2,components.o3,components.so2,components.pm2_5,components.pm10,components.nh3,coord.lon,coord.lat,extraction_date_time,city_name
0,2023-11-06 15:52:19+08:00,2,337.12,0.10,1.95,76.53,1.10,18.51,20.02,2.15,119.9811,16.1561,2023-11-06 15:52:19.564737+08:00,Alaminos
1,2023-11-06 15:52:20+08:00,2,343.80,0.36,5.01,60.80,3.61,8.12,9.62,3.20,120.5833,15.1500,2023-11-06 15:52:21.211334+08:00,Angeles City
2,2023-11-06 15:52:22+08:00,1,240.33,0.10,1.41,51.50,0.41,1.28,1.65,1.20,121.1900,15.6181,2023-11-06 15:52:22.835422+08:00,Antipolo
3,2023-11-06 15:52:23+08:00,1,181.91,0.05,0.41,37.55,1.33,1.12,1.28,0.36,123.5085,12.2285,2023-11-06 15:52:24.194722+08:00,Bacolod
4,2023-11-06 15:52:25+08:00,5,1001.36,0.05,5.44,254.63,28.85,102.82,131.67,9.75,120.9290,14.4590,2023-11-06 15:52:25.758546+08:00,Bacoor


In [4]:
features = [
    'components.co', 'components.no', 'components.no2',
    'components.o3', 'components.so2', 'components.pm2_5',
    'components.pm10', 'components.nh3'
]
X = df[features]
y = df['main.aqi']

In [5]:
# Encode target to 0-4
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [6]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

In [9]:
#Model Training
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_balanced, y_train_balanced)

RandomForestClassifier(n_jobs=-1, random_state=42)

In [10]:
y_pred = rf_model.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)

print(f"Accuracy: {acc*100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=[str(c) for c in label_encoder.classes_]))

Accuracy: 99.99%

Classification Report:
              precision    recall  f1-score   support

           1       1.00      1.00      1.00     50865
           2       1.00      1.00      1.00     30719
           3       1.00      1.00      1.00      6469
           4       1.00      1.00      1.00      1554
           5       1.00      1.00      1.00      1564

    accuracy                           1.00     91171
   macro avg       1.00      1.00      1.00     91171
weighted avg       1.00      1.00      1.00     91171



In [11]:
# Sample 1: Low pollutants (Should be AQI 1 or 2)
# Sample 2: Extremely high PM2.5 and PM10 (Should be AQI 4 or 5)

new_data = {
    'components.co': [200.0, 1000.5],
    'components.no': [0.1, 5.0],
    'components.no2': [1.5, 35.0],
    'components.o3': [40.0, 150.0],
    'components.so2': [0.5, 20.0],
    'components.pm2_5': [5.0, 150.0],   # PM2.5 is a huge driver of AQI
    'components.pm10': [8.0, 200.0],
    'components.nh3': [1.0, 10.0]
}

sample_df = pd.DataFrame(new_data)
sample_scaled = scaler.transform(sample_df)
prediction_encoded = rf_model.predict(sample_scaled)
final_aqi_prediction = label_encoder.inverse_transform(prediction_encoded)

for i, aqi in enumerate(final_aqi_prediction):
    print(f"Sample {i+1} Predicted AQI: {aqi}")

Sample 1 Predicted AQI: 1
Sample 2 Predicted AQI: 5


In [12]:
import joblib
import os

# 1. Create a folder named 'model' to match your Flask app's paths
os.makedirs('model', exist_ok=True)

# 2. Save the Random Forest model (saving it as 'best_model.pkl')
joblib.dump(rf_model, 'model/best_model.pkl')

# 3. Save the Scaler
joblib.dump(scaler, 'model/scaler.pkl')

# 4. Save the Label Encoder (Highly recommended so Flask knows how to translate 0-4 back to AQI 1-5)
joblib.dump(label_encoder, 'model/label_encoder.pkl')

print("PKL files successfully saved in the 'model' folder!")

PKL files successfully saved in the 'model' folder!
